# 阶段 6：Colab 闭环（PyTorch 原精度）

和 Jetson 上的虚拟时间闭环跑**同一组点**：3 台服务器、N = 20/40/60、种子 0 和 1、每点 1000 槽。仿真器、任务、种子、打分都是同一份代码（从 Mac 打包上传），只是作答的换成 Colab 上合并了 LoRA 的 bf16 模型。两边结果的差别，就是部署带来的变化（量化到 Q8_0、换成 llama.cpp）。

**只改第 3 格的 `RUN`**，然后「代码执行程序 → 全部运行」：

| `RUN` 填 | 做什么 | G4 上预计（估算） |
|---|---|---|
| `students` | 6 个学生依次跑，每个 6 个点 | 1–1.5 小时 |
| 一个模型键，如 `ps_soft_1.5b` | 只跑这一个 | 0.5B 约 10 分钟，1.5B 约 15 分钟 |
| `comllm_7b_repro` | 复现的 COMLLM-7B（论文格式，生成回答），等 7B 训练完再跑 | 1–2 小时 |
| `grpo_7b_repro` | 复现的 GRPO-7B（不带 LACS 的对照，可选） | 1–2 小时 |

预计时长的依据（估算）：同一个模型的 6 个点同时跑、拼成一批作答，前向次数约等于 N = 60 那个点的决策数（约 1.8 万次）；0.5B 每次前向约 20 ms，1.5B 约 30 ms，再加上载入模型。7B 每次要生成几个字，按每批 0.2–0.4 秒估。

跑之前：运行时选 **G4**（没有就 A100 或 L4）。Drive 里要有：
- `MyDrive/COMLLM/s6_code/`：Mac 上 `python -m eval.colab_loop --pack` 生成的 `data/s6_code/` 整个文件夹（code.zip 和 meta.json）
- `MyDrive/COMLLM/s3_runs/` 下 6 个学生的运行目录（训练 notebook 存的 `adapter/` 和 `config.json`）
- 跑 7B 时还要：`MyDrive/COMLLM/reference_model/` 下原作者的 SFT-7B，和 7B 复现 notebook 存的 `MyDrive/COMLLM/s7_runs/lacs/lora/`（GRPO-7B 是 `s7_runs/nolacs/lora/`）

**结果**在 Drive 的 `MyDrive/COMLLM/s6_runs/colab_loop/results.csv`。断线了重新「全部运行」会接着跑，跑完的点自动跳过。

**止损线**：
- 第 4 格的核对点和 Mac 上算的不一样，马上停：说明 Colab 上的仿真器和 Mac 上的不是同一个，后面的数不能比。
- 第一个模型跑完会打印用时。超过上表估算的 2 倍就停下，把输出截图发到对话里。
- 任何一格报错就停，截图发到对话里。
- 跑完「代码执行程序 → 断开连接并删除运行时」，停止扣额度。

In [ ]:
# 第 1 格：安装固定版本的库（和阶段 3 训练 notebook 同样的版本）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")

In [ ]:
# 第 2 格：检查 GPU（要支持 bf16；T4 不支持）
import os, time, json, csv, hashlib, zipfile, glob
from datetime import datetime, timezone
import torch

assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选 G4 后再运行"
assert torch.cuda.is_bf16_supported(), "这块 GPU 不支持 bf16（T4 就不支持）：换成 G4、A100 或 L4"
DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name()
subprocess.run(["nvidia-smi"], check=False)
print("GPU:", GPU_NAME, "| torch", torch.__version__)

In [ ]:
# 第 3 格：这次跑哪些模型。每次只改 RUN 这一行
RUN = "students"      # students = 6 个学生依次跑；也可以写一个模型键，如 "ps_soft_1.5b"、"comllm_7b_repro"

# ---- 下面不用改 ----
DRIVE = "/content/drive/MyDrive/COMLLM"
CODE_DIR = f"{DRIVE}/s6_code"
OUT = f"{DRIVE}/s6_runs/colab_loop"
BASES = {   # 基座: (Hugging Face 上的模型, 钉死的提交号，见 docs/versions.md)
    "0.5B": ("Qwen/Qwen2.5-0.5B-Instruct", "7ae557604adf67be50417f59c2c2f167def9a775"),
    "1.5B": ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
}
MODELS = {  # 模型键: 基座、LoRA 在哪个运行目录（取前缀相同里最新的一个）、提示词格式
    "main_0.5b":    {"base": "0.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "main_0.5b_",    "fmt": "compact"},
    "soft_1.5b":    {"base": "1.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "soft_1.5b_",    "fmt": "compact"},
    "hard_1.5b":    {"base": "1.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "hard_1.5b_",    "fmt": "compact"},
    "ps_0.5b":      {"base": "0.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "ps_0.5b_",      "fmt": "compact"},
    "ps_soft_1.5b": {"base": "1.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "ps_soft_1.5b_", "fmt": "compact"},
    "ps_hard_1.5b": {"base": "1.5B", "runs": f"{DRIVE}/s3_runs", "prefix": "ps_hard_1.5b_", "fmt": "compact"},
    # 复现的 7B：基座是原作者的 SFT-7B（reference_model/ 下 hidden_size = 3584 的那个），LoRA 是 7B 复现 notebook 存的。
    # 论文格式、贪心生成最多 16 个 token；读法和 7B 开环评测一样，在回答里找第一个动作名（search）
    "comllm_7b_repro": {"base": "sft_7b", "adapter": f"{DRIVE}/s7_runs/lacs/lora", "fmt": "paper",
                        "parse": "search", "max_new_tokens": 16},
    "grpo_7b_repro":   {"base": "sft_7b", "adapter": f"{DRIVE}/s7_runs/nolacs/lora", "fmt": "paper",
                        "parse": "search", "max_new_tokens": 16},
}

In [ ]:
# 第 4 格：挂 Drive，解压仓库代码，核对它和 Mac 上是同一份
#   1. code.zip 的校验值要和 meta.json 一样
#   2. 解压到 /content/code，加进 import 路径
#   3. 核对点：单步公式在 3 台、N = 20、种子 0 上跑一遍，平均延迟必须和 Mac 上打包时算的一模一样
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

with open(f"{CODE_DIR}/meta.json", encoding="utf-8") as f:
    META = json.load(f)
h = hashlib.md5()
with open(f"{CODE_DIR}/code.zip", "rb") as f:
    h.update(f.read())
assert h.hexdigest() == META["md5"], "code.zip 和 meta.json 对不上：Mac 上重新 --pack，整个 s6_code 文件夹重新上传"
with zipfile.ZipFile(f"{CODE_DIR}/code.zip") as z:
    z.extractall("/content/code")
if "/content/code" not in sys.path:
    sys.path.insert(0, "/content/code")
os.chdir("/content/code")

from eval import colab_loop as CL
from eval.jetson_open_loop import COLAB_NAME
importlib.reload(CL)
v = CL.check_value()
assert v == META["check_mean_delay"], f"核对点不一致：Colab {v}，Mac {META['check_mean_delay']}。先停下，把这一行截图发到对话里"
print(f"代码版本 {META['git']}，{len(META['files'])} 个文件 ✓；核对点平均延迟 {v} 和 Mac 上一样 ✓")

In [ ]:
# 第 5 格：找 LoRA、载入模型（合并 LoRA，bf16）
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


def find_adapter(key):
    """取前缀相同的运行目录里最新的一个，里面要有 adapter/ 和 config.json，且 config 里的名字对得上。"""
    m = MODELS[key]
    dirs = sorted(d for d in glob.glob(os.path.join(m["runs"], m["prefix"] + "*"))
                  if os.path.isdir(os.path.join(d, "adapter")))
    assert dirs, f"{m['runs']} 下没有 {m['prefix']}* 的运行目录（要有 adapter/）：先跑训练 notebook"
    run = dirs[-1]
    with open(os.path.join(run, "config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    assert cfg.get("name") == COLAB_NAME[key], f"{run} 的 config 名字是 {cfg.get('name')}，应该是 {COLAB_NAME[key]}"
    return run


def find_sft_7b():
    """reference_model/ 下 config.json 里 hidden_size = 3584 的文件夹就是原作者的 SFT-7B。"""
    for d in sorted(glob.glob(f"{DRIVE}/reference_model/*")):
        p = os.path.join(d, "config.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                if json.load(f).get("hidden_size") == 3584:
                    return d
    raise AssertionError("reference_model/ 下没找到 SFT-7B（hidden_size = 3584）：先把它从 Mac 传到 Drive")


def load_model(key):
    """返回 (分词器, 合并好 LoRA 的模型, LoRA 来源)。学生：基座从 Hugging Face 取；7B：基座是 Drive 上的 SFT-7B。"""
    m = MODELS[key]
    if m["base"] == "sft_7b":
        base_path, rev = find_sft_7b(), None
        assert os.path.exists(os.path.join(m["adapter"], "adapter_config.json")), f"{m['adapter']} 下没有 LoRA：7B 还没训完"
        adapter, run = m["adapter"], m["adapter"]
    else:
        base_path, rev = BASES[m["base"]]
        run = find_adapter(key)
        adapter = os.path.join(run, "adapter")
    tok = AutoTokenizer.from_pretrained(base_path, revision=rev)
    base = AutoModelForCausalLM.from_pretrained(base_path, revision=rev, dtype=torch.bfloat16).to(DEVICE)
    model = PeftModel.from_pretrained(base, adapter).merge_and_unload()
    model.eval()
    return tok, model, run


KEYS = CL.STUDENTS if RUN == "students" else [RUN]
for key in KEYS:
    print(key, "→", MODELS[key].get("adapter") or find_adapter(key))

In [ ]:
# 第 6 格：逐个模型跑 6 个点（同一个模型的 6 个点同时跑，拼批作答）
import transformers, peft

for key in KEYS:
    t0 = time.time()
    print(f"\n===== {key} =====", flush=True)
    tok, model, run = load_model(key)
    if MODELS[key]["fmt"] == "compact":
        decide = CL.make_compact_decider(model, tok, DEVICE)
    else:
        decide = CL.make_paper_decider(model, tok, DEVICE, max_new_tokens=MODELS[key].get("max_new_tokens", 20),
                                       parse=MODELS[key].get("parse", "strict"))
    with open(f"{OUT}/sweep_config.json", "a", encoding="utf-8") as f:
        f.write(json.dumps({"model_key": key, "adapter_run": run, "base": BASES.get(MODELS[key]["base"], MODELS[key]["base"]),
                            "fmt": MODELS[key]["fmt"], "code_git": META["git"], "code_md5": META["md5"],
                            "gpu": GPU_NAME, "torch": torch.__version__, "transformers": transformers.__version__,
                            "peft": peft.__version__, "platform": "colab",
                            "started_utc": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")},
                           ensure_ascii=False) + "\n")
    CL.run_points(CL.points_for(key), decide, f"{OUT}/results.csv", sha=os.path.basename(run.rstrip("/")))
    del model
    torch.cuda.empty_cache()
    print(f"{key} 用时 {(time.time() - t0) / 60:.1f} 分钟（含载入模型）", flush=True)

In [ ]:
# 第 7 格：汇总（每个模型、每个 N：两个种子的平均）
rows = []
with open(f"{OUT}/results.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append(r)
cell = {}
for r in rows:
    cell.setdefault((r["policy"], int(r["n"])), []).append(r)
print("| 模型 | N | 种子数 | 平均延迟（槽） | 丢弃率 % | 回退次数 | 和单步最优一致 % |")
print("|---|---|---|---|---|---|---|")
for (pol, n) in sorted(cell):
    rs = cell[(pol, n)]
    d = sum(float(r["mean_delay"]) for r in rs) / len(rs)
    x = sum(float(r["drop_rate"]) for r in rs) / len(rs)
    fb = sum(int(r["fallback"]) for r in rs)
    ag = sum(float(r["agree_greedy"]) for r in rs) / len(rs)
    print(f"| {pol} | {n} | {len(rs)} | {d:.3f} | {100 * x:.1f} | {fb} | {100 * ag:.1f} |")

## 跑完之后

1. 把第 7 格的表截图发到对话里。
2. 把 Drive 上的 `MyDrive/COMLLM/s6_runs/colab_loop/` 整个文件夹下载到 Mac，放进仓库的 `runs/`，改名成 `<UTC 时间>_s6_colab_loop`（时间写下载那一刻的，格式和其他运行目录一样，如 `20261112T080000Z_s6_colab_loop`）。
3. 菜单「代码执行程序 → 断开连接并删除运行时」，停止扣额度。